# Hyperparameter Tuning — EX15 FleetBrokenGlobal

Sequential tuning notebook. Each phase has a **lock-in cell** — set that value before running the next phase.

| Phase | Parameter | Evaluated via |
|-------|-----------|---------------|
| 1 | Learning rate α | Training convergence curves (VFA-only) |
| 2 | Discount factor γ | Training convergence curves (VFA-only) |
| 3 | Reward weights (ws, wc) | Train + full RA evaluation |
| 4 | Rollout: H, S, R | Full RA evaluation, coordinate descent |

> **Note on Phases 1–2:** Training curves (SL + weight stability) are sufficient for α and γ selection.
> This is consistent with how the feature ablation was conducted and avoids 5×3 = 15 expensive RA runs.
> Run one validation RA evaluation after locking both α and γ before moving to Phase 3.

In [ ]:
import os, sys, re, shutil, tempfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fomosim_matplotlib"))

# ── Workspace root ────────────────────────────────────────────────────────
WORKSPACE_ROOT = Path.cwd()
while WORKSPACE_ROOT.name != "FOMOsim" and WORKSPACE_ROOT.parent != WORKSPACE_ROOT:
    WORKSPACE_ROOT = WORKSPACE_ROOT.parent

# ── LaTeX styling (mirrors plot_ablation_study.py) ────────────────────────
if shutil.which("latex"):
    plt.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "text.latex.preamble": r"\usepackage[T1]{fontenc} \usepackage{mlmodern}",
    })
else:
    plt.rcParams.update({"text.usetex": False, "font.family": "serif"})

# Adjust these to taste
TITLE_FONTSIZE  = 13
AXIS_FONTSIZE   = 12
LEGEND_FONTSIZE = 9
PLOT_DPI        = 150

COLORS = [
    "#344E41", "#D1495B", "#526ECA", "#EDAE49", "#84A579",
    "#7B3F5E", "#4A90A4", "#C67C3E", "#3D7068", "#A44A3F",
]

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print(f"Workspace: {WORKSPACE_ROOT}")

In [ ]:
# ── Helpers: parse folder names produced by run_ablation_study.py ─────────
_FOLDER_RE    = re.compile(r"^(.+)_(?:alpha|adam|sgd)_([0-9]+(?:\.[0-9]+)?)(?:_(.*))?$")
_TIMESTAMP_RE = re.compile(r"(?:^|_)\d{8}_\d{6}(?:\s+copy)?$")
_GAMMA_RE     = re.compile(r"_g([0-9]+p[0-9]+)(?:_|$)")
_WS_RE        = re.compile(r"_ws(m?[0-9]+p[0-9]+)(?:_|$)")
_WC_RE        = re.compile(r"_wc(m?[0-9]+p[0-9]+)(?:_|$)")

def _token_to_float(t: str) -> float:
    return float(t.replace("p", ".").replace("m", "-"))

_META_COLS = {
    "episode", "service_level", "alpha", "alpha_initial", "epsilon",
    "epsilon_initial", "starvations", "gamma", "transition_update_interval",
    "transition_batch_updates", "remainder_batch_updates",
    "batch_updates_this_episode", "long_congestions", "short_congestions",
    "total_trips", "bike_departures", "bike_arrivals", "total_onsite_repairs",
    "total_depot_pickups", "total_depot_deliveries", "total_depot_visits",
    "total_functional_pickups", "total_functional_deliveries",
    "broken_ratio_start_onsite", "broken_ratio_start_depot",
    "functional_ratio_start", "broken_ratio_end_onsite",
    "broken_ratio_end_depot", "functional_ratio_end",
    "new_breakdowns_onsite", "new_breakdowns_depot",
    "restored_onsite", "restored_depot",
}


def load_training_runs(study_dir, *, group_by: str = "alpha", smooth_window: int = 30):
    """
    Scan study_dir for weights_evolution CSVs.
    group_by: 'alpha' | 'gamma' | 'reward'
    Returns dict[key → list[run_dict]]
    """
    study_dir = Path(study_dir) if Path(str(study_dir)).is_absolute() else WORKSPACE_ROOT / study_dir
    groups = defaultdict(list)

    for folder in sorted(study_dir.iterdir()):
        if not folder.is_dir():
            continue
        m = _FOLDER_RE.match(folder.name)
        if not m:
            continue
        base, alpha, suffix = m.group(1), m.group(2), m.group(3) or ""
        suffix = _TIMESTAMP_RE.sub("", suffix).strip("_")

        if group_by == "alpha":
            key = alpha
        elif group_by == "gamma":
            gm = _GAMMA_RE.search(suffix)
            key = gm.group(1) if gm else alpha
        elif group_by == "reward":
            ws = _WS_RE.search(suffix)
            wc = _WC_RE.search(suffix)
            key = (f"ws={_token_to_float(ws.group(1)):+.1f}, wc={_token_to_float(wc.group(1)):+.1f}"
                   if ws and wc else "default")
        else:
            key = folder.name

        for csv in sorted(folder.glob("*_weights_evolution.csv")):
            sm = re.search(r"seed(\d+)", csv.name)
            seed = sm.group(1) if sm else "?"
            df = pd.read_csv(csv)
            if "episode" not in df.columns or "service_level" not in df.columns:
                continue
            df["episode"]       = pd.to_numeric(df["episode"],       errors="coerce")
            df["service_level"] = pd.to_numeric(df["service_level"], errors="coerce")
            df = df.dropna(subset=["episode", "service_level"])
            if df.empty:
                continue
            w_cols = [c for c in df.columns if c not in _META_COLS]
            w_df   = df[w_cols].apply(pd.to_numeric, errors="coerce")
            sl_sm  = pd.Series(df["service_level"].values).rolling(smooth_window, min_periods=1).mean().values
            groups[key].append({
                "seed": seed, "folder": folder.name, "alpha": alpha,
                "episodes": df["episode"].values,
                "sl_raw": df["service_level"].values, "sl": sl_sm,
                "weights": w_df.values, "features": w_cols,
            })

    return dict(groups)


def _conv_metrics(sl: np.ndarray, last_n: int = 30) -> dict:
    peak  = float(np.max(sl))
    final = float(np.mean(sl[-last_n:]))
    std   = float(np.std(sl[-last_n:]))
    thr   = 0.95 * peak
    conv  = next((i for i in range(len(sl) - 10) if np.all(sl[i:i+10] >= thr)), None)
    return {"peak": peak, "final": final, "std": std, "conv_ep": conv}


def plot_sl_comparison(groups: dict, title: str, param_label: str = r"$\alpha$",
                       last_n: int = 30, figsize=(12, 6)) -> plt.Figure:
    """SL convergence: one line per group (mean ± 1 std across seeds)."""
    fig, ax = plt.subplots(figsize=figsize)
    cmap = {k: COLORS[i % len(COLORS)] for i, k in enumerate(sorted(groups))}

    for key in sorted(groups):
        runs = groups[key]
        if not runs:
            continue
        n = min(len(r["sl"]) for r in runs)
        sls  = np.array([r["sl"][:n] for r in runs])
        eps  = runs[0]["episodes"][:n]
        mean = sls.mean(axis=0)
        std  = sls.std(axis=0)
        col  = cmap[key]
        try:
            v = float(key.replace("p", ".").replace("m", "-")) if "p" in key else float(key)
            lbl = rf"{param_label}$={v:g}$  (peak={mean.max():.4f}, final={mean[-last_n:].mean():.4f})"
        except ValueError:
            lbl = rf"{param_label}$={key}$  (peak={mean.max():.4f}, final={mean[-last_n:].mean():.4f})"

        ax.plot(eps, mean, linewidth=2.0, color=col, alpha=0.85, label=lbl)
        ax.fill_between(eps, mean - std, mean + std, color=col, alpha=0.12)
        trend = np.poly1d(np.polyfit(eps, mean, 1))(eps)
        ax.plot(eps, trend, linestyle="--", linewidth=2.0, color=col, alpha=0.35)

    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Training episode", fontsize=AXIS_FONTSIZE)
    ax.set_ylabel(r"Service level (30-ep moving avg)", fontsize=AXIS_FONTSIZE)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=LEGEND_FONTSIZE, frameon=True)
    plt.tight_layout()
    return fig


def plot_weight_evolution(runs: list, title: str, figsize=(12, 5)) -> plt.Figure:
    """Mean weight trajectories across seeds for one group."""
    if not runs:
        return None
    n    = min(len(r["weights"]) for r in runs)
    feat = runs[0]["features"]
    W    = np.nanmean(np.array([r["weights"][:n, :len(feat)] for r in runs], dtype=float), axis=0)
    eps  = runs[0]["episodes"][:n]

    fig, ax = plt.subplots(figsize=figsize)
    for i, f in enumerate(feat):
        ax.plot(eps, W[:, i], linewidth=2.0, alpha=0.85,
                color=COLORS[i % len(COLORS)], label=f"{f} ({W[-1, i]:+.3f})")
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--", alpha=0.4)
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.set_xlabel("Episode", fontsize=AXIS_FONTSIZE)
    ax.set_ylabel(r"Weight ($\theta$)", fontsize=AXIS_FONTSIZE)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=10)
    plt.tight_layout()
    return fig


def convergence_table(groups: dict, param_name: str = "alpha", last_n: int = 30):
    """Styled convergence table for all groups."""
    rows = []
    for key in sorted(groups):
        runs = groups[key]
        if not runs:
            continue
        n    = min(len(r["sl"]) for r in runs)
        sls  = np.array([r["sl"][:n] for r in runs])
        mean = sls.mean(axis=0)
        seed_finals = sls[:, -last_n:].mean(axis=1)
        m = _conv_metrics(mean, last_n)
        rows.append({
            param_name:              key,
            "N seeds":               len(runs),
            "Peak SL":               m["peak"],
            "Final SL (mean)": m["final"],
            "Final SL (std, seeds)": float(np.std(seed_finals)),
            "Convergence ep.": m["conv_ep"] if m["conv_ep"] is not None else ">300",
        })
    df = pd.DataFrame(rows).set_index(param_name)
    return (
        df.style
          .format({"Peak SL": "{:.4f}", "Final SL (mean)": "{:.4f}",
                   "Final SL (std, seeds)": "{:.4f}"})
          .background_gradient(subset=["Final SL (mean)"], cmap="Blues")
          .set_caption(f"Convergence metrics grouped by {param_name}")
    )

print("Training helpers loaded.")

In [ ]:
# ── Helpers: evaluation results (results.csv from run_logs/) ──────────────

EVAL_METRIC_LABELS = {
    "service_level":        r"Service level",
    "starvations":          r"Starvations",
    "congestions":          r"Congestions",
    "functional_ratio_end": r"Func. ratio end (FR)",
    "maintenance_pct":      r"Maintenance action %",
    "total_depot_visits":   r"Depot visits",
}

def _strip_suffix(name: str) -> str:
    name = re.sub(r"_seed\d+.*$", "", name)
    name = re.sub(r"_TD_W\d+.*$", "", name)
    name = re.sub(r"_OS_W\d+.*$", "", name)
    return name.strip("_")

def _label_from_path(csv_path: Path) -> str:
    folder = csv_path.parent.name
    if folder.startswith("Hybrid_") or folder.startswith("hybrid_"):
        return "Hybrid: " + _strip_suffix(re.sub(r"^[Hh]ybrid_", "", folder))
    if folder.startswith("VFA_") or folder.startswith("vfa_"):
        return "VFA: "    + _strip_suffix(re.sub(r"^[Vv][Ff][Aa]_", "", folder))
    if "greedy" in folder.lower():    return "Greedy maintenance"
    if "nothing" in folder.lower():   return "Do nothing"
    return folder

def load_eval_results(run_log_dirs: list, label_override: dict | None = None) -> pd.DataFrame:
    """
    Load results.csv files from a list of run_log paths.
    label_override: {run_log_path_str: display_label} — use for Phase 4 to name configs clearly.
    """
    rows = []
    for raw in run_log_dirs:
        path = Path(raw) if Path(str(raw)).is_absolute() else WORKSPACE_ROOT / raw
        if path.is_file() and path.name == "results.csv":
            csv_files = [path]
        elif path.is_dir():
            csv_files = sorted(path.rglob("results.csv"))
        else:
            print(f"Warning: no results.csv found at {raw}")
            csv_files = []
        for csv_path in csv_files:
            lbl = (label_override or {}).get(str(raw)) or _label_from_path(csv_path)
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f"Warning: {csv_path}: {e}")
                continue
            df["label"]  = lbl
            df["source"] = str(csv_path.relative_to(WORKSPACE_ROOT))
            rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def _add_derived(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    maint = df.get("total_fixed_on_site", 0).fillna(0) + df.get("total_picked_up_to_depot", 0).fillna(0)
    reb   = df.get("total_functional_pickups", 0).fillna(0) + df.get("total_functional_deliveries", 0).fillna(0)
    df["maintenance_pct"] = np.where((maint + reb) > 0, maint / (maint + reb), np.nan)
    return df

def eval_summary_table(df: pd.DataFrame, metrics: list, sort_by: str = "service_level"):
    """Styled wide table: mean ± std per label × metric, sorted by sort_by."""
    df = _add_derived(df)
    metrics = [m for m in metrics if m in df.columns]
    rows = []
    for lbl, grp in df.groupby("label"):
        row = {"Label": lbl, "N": int(len(grp))}
        for m in metrics:
            row[f"{m}_mean"] = grp[m].mean()
            row[f"{m}_std"]  = grp[m].std()
        rows.append(row)
    out = pd.DataFrame(rows).set_index("Label")
    if f"{sort_by}_mean" in out.columns:
        out = out.sort_values(f"{sort_by}_mean", ascending=False)
    fmt = {}
    rename = {}
    for m in metrics:
        lbl_m = EVAL_METRIC_LABELS.get(m, m.replace("_", " ").title())
        rename[f"{m}_mean"] = f"{lbl_m} (mean)"
        rename[f"{m}_std"]  = f"{lbl_m} (std)"
        if m in ("service_level", "functional_ratio_end", "maintenance_pct"):
            fmt[f"{m}_mean"] = "{:.4f}"; fmt[f"{m}_std"] = "{:.4f}"
        else:
            fmt[f"{m}_mean"] = "{:.1f}";  fmt[f"{m}_std"] = "{:.1f}"
    cols = [c for c in out.columns if c != "N"]
    return (
        out[cols].rename(columns=rename)
                 .style
                 .format({rename[k]: v for k, v in fmt.items() if k in rename})
                 .background_gradient(
                     subset=[rename.get(f"{sort_by}_mean", f"{sort_by}_mean")],
                     cmap="Blues"
                 )
                 .set_caption("Evaluation summary (mean \u00b1 std across seeds)")
    )

def plot_eval_bars(df: pd.DataFrame, metrics: list, sort_by: str = "service_level",
                   figsize=None) -> plt.Figure:
    df = _add_derived(df)
    metrics = [m for m in metrics if m in df.columns]
    if not metrics:
        print("No valid metrics."); return None
    summary = df.groupby("label")[metrics].agg(["mean", "std"])
    order   = summary[(sort_by, "mean")].sort_values(ascending=False).index if sort_by in metrics else summary.index
    y       = np.arange(len(order))
    fig, axes = plt.subplots(1, len(metrics),
                             figsize=figsize or (5.5 * len(metrics), max(4.0, 0.45 * len(order))),
                             squeeze=False)
    for ax, m in zip(axes[0], metrics):
        means = summary.loc[order, (m, "mean")].to_numpy(dtype=float)
        stds  = summary.loc[order, (m, "std")].fillna(0).to_numpy(dtype=float)
        ax.barh(y, means, xerr=stds, color="#4C78A8", alpha=0.88)
        ax.invert_yaxis()
        ax.set_title(EVAL_METRIC_LABELS.get(m, m.replace("_", " ").title()), fontsize=AXIS_FONTSIZE)
        ax.grid(axis="x", alpha=0.25)
        ax.set_yticks(y, [str(l).replace("_", " ") for l in order] if m == metrics[0] else [""]*len(order))
        for yi, v in zip(y, means):
            if not np.isnan(v):
                ax.text(v, yi, f"  {v:.4f}" if abs(v) < 10 else f"  {v:.0f}", va="center", fontsize=9)
    fig.tight_layout()
    return fig

print("Evaluation helpers loaded.")

---
## Phase 1 — Learning Rate α

**Before running this section**, train EX15 with the candidate α values:

```bash
python policies/sjovik_sund/ablation_study/run_ablation_study.py \
  --experiments EX15_FleetBrokenGlobal \
  --alphas 0.005 0.01 0.05 0.1 0.5 \
  --seeds 1000 \
  --episodes 300 \
  --output_dir tuning/phase1_alpha \
  --gamma 0.97 \
  --weight_starvation -1.0 \
  --weight_congestion -1.0 \
  --weight_fleet_degradation 0.0 \
  --use_bias_feature \
  --initial_bias -2.5 \
  --transition_update_interval 100 \
  --use_terminal_update \
  --use_feature_scale_diagnostics \
  --diagnostic_every_n_episodes 25 \
  --epsilon_start 0.0 \
  --epsilon_end 0.0
```

In [ ]:
# ── Phase 1 config ────────────────────────────────────────────────────────
PHASE1_DIR    = "models/tuning/phase1_alpha"   # output_dir passed to run_ablation_study
PHASE1_LAST_N = 30                             # episodes averaged for "final SL"

groups_alpha = load_training_runs(PHASE1_DIR, group_by="alpha", smooth_window=30)
print(f"Found {len(groups_alpha)} alpha group(s):")
for k, v in sorted(groups_alpha.items()):
    print(f"  \u03b1={k}: {len(v)} seed(s) \u2014 {[r['seed'] for r in v]}")

In [ ]:
# ── Phase 1: SL convergence comparison ───────────────────────────────────
fig = plot_sl_comparison(
    groups_alpha,
    title=r"Phase 1 --- Service level convergence by $\alpha$ (EX15 FleetBrokenGlobal)",
    param_label=r"$\alpha$",
    last_n=PHASE1_LAST_N,
)
plt.show()

In [ ]:
# ── Phase 1: Weight evolution per α ──────────────────────────────────────
for key in sorted(groups_alpha):
    runs = groups_alpha[key]
    try:
        v = float(key)
        title = rf"Phase 1 --- Weight evolution, $\alpha={v:g}$ (mean over {len(runs)} seed(s))"
    except ValueError:
        title = f"Phase 1 \u2014 Weight evolution, \u03b1={key}"
    fig = plot_weight_evolution(runs, title=title)
    if fig:
        plt.show()

In [ ]:
# ── Phase 1: Convergence metrics table ───────────────────────────────────
convergence_table(groups_alpha, param_name=r"$\alpha$", last_n=PHASE1_LAST_N)

In [ ]:
# ── Phase 1: Lock α ──────────────────────────────────────────────────────
LOCKED_ALPHA = 0.05   # ← SET THIS after reviewing results above
print(f"Locked:  \u03b1 = {LOCKED_ALPHA}")

---
## Phase 2 — Discount Factor γ

**Before running this section**, train with the locked α:

```bash
python policies/sjovik_sund/ablation_study/run_ablation_study.py \
  --experiments EX15_FleetBrokenGlobal \
  --alphas <LOCKED_ALPHA> \
  --gammas 0.95 0.97 0.99 \
  --seeds 1000 \
  --episodes 300 \
  --output_dir tuning/phase2_gamma \
  --weight_starvation -1.0 \
  --weight_congestion -1.0 \
  --weight_fleet_degradation 0.0 \
  --use_bias_feature \
  --initial_bias -2.5 \
  --transition_update_interval 100 \
  --use_terminal_update \
  --use_feature_scale_diagnostics \
  --diagnostic_every_n_episodes 25 \
  --epsilon_start 0.0 \
  --epsilon_end 0.0
```

In [ ]:
# ── Phase 2 config ────────────────────────────────────────────────────────
PHASE2_DIR    = "models/tuning/phase2_gamma"
PHASE2_LAST_N = 30

groups_gamma = load_training_runs(PHASE2_DIR, group_by="gamma", smooth_window=30)
print(f"Found {len(groups_gamma)} gamma group(s):")
for k, v in sorted(groups_gamma.items()):
    try:
        gval = _token_to_float(k)
        print(f"  \u03b3={gval:g}: {len(v)} seed(s)")
    except Exception:
        print(f"  key={k}: {len(v)} seed(s)")

In [ ]:
# ── Phase 2: SL convergence ───────────────────────────────────────────────
fig = plot_sl_comparison(
    groups_gamma,
    title=r"Phase 2 --- Service level convergence by $\gamma$ (EX15, locked $\alpha$)",
    param_label=r"$\gamma$",
    last_n=PHASE2_LAST_N,
)
plt.show()

In [ ]:
# ── Phase 2: Weight evolution per γ ──────────────────────────────────────
for key in sorted(groups_gamma):
    runs = groups_gamma[key]
    try:
        v = _token_to_float(key)
        title = rf"Phase 2 --- Weight evolution, $\gamma={v:g}$"
    except Exception:
        title = f"Phase 2 \u2014 Weight evolution, \u03b3={key}"
    fig = plot_weight_evolution(runs, title=title)
    if fig:
        plt.show()

In [ ]:
# ── Phase 2: Convergence table ────────────────────────────────────────────
convergence_table(groups_gamma, param_name=r"$\gamma$", last_n=PHASE2_LAST_N)

In [ ]:
# ── Phase 2: Lock γ ───────────────────────────────────────────────────────
LOCKED_GAMMA = 0.97   # ← SET THIS after reviewing results above
print(f"Locked:  \u03b3 = {LOCKED_GAMMA}")

---
## Phase 3 — Reward Weights (ws, wc)

Run three separate training jobs (one per variant), then evaluate each with
`run_simulation_ingvild.py --policy hybrid`.

```bash
# Variant A: equal weights (baseline)
python policies/sjovik_sund/ablation_study/run_ablation_study.py \
  --experiments EX15_FleetBrokenGlobal \
  --alphas <LOCKED_ALPHA> \
  --gammas <LOCKED_GAMMA> \
  --weight_starvation -1.0 \
  --weight_congestion -1.0 \
  --weight_fleet_degradation 0.0 \
  --seeds 1000 \
  --episodes 300 \
  --output_dir tuning/phase3_reward \
  --use_bias_feature \
  --initial_bias -2.5 \
  --transition_update_interval 100 \
  --use_terminal_update \
  --use_feature_scale_diagnostics \
  --diagnostic_every_n_episodes 25 \
  --epsilon_start 0.0 \
  --epsilon_end 0.0

# Variant B: starvation-downweighted  (change --weight_starvation -0.7)
# Variant C: congestion-downweighted  (change --weight_congestion -0.7)
```

Then evaluate each trained model:

```bash
python policies/sjovik_sund/run_simulation_ingvild.py \
  --policy hybrid --vfa-model <PKL_PATH> \
  --lookahead <LOCKED_H> --num-scenarios <LOCKED_S> --n-routing <LOCKED_R> \
  --warmup-days 7 --duration 504 --nsims 10 --seed 42 \
  --log-files results daily
```

Point `PHASE3_RUN_LOGS` to the resulting run log folders below.

In [ ]:
# ── Phase 3 config ────────────────────────────────────────────────────────
# Map run_log path → display label for each reward variant
PHASE3_RUN_LOGS = {
    "run_logs/tuning/phase3_ws-1.0_wc-1.0": r"ws=$-1.0$, wc=$-1.0$ (equal)",
    "run_logs/tuning/phase3_ws-0.7_wc-1.0": r"ws=$-0.7$, wc=$-1.0$ (starv. down)",
    "run_logs/tuning/phase3_ws-1.0_wc-0.7": r"ws=$-1.0$, wc=$-0.7$ (cong. down)",
}
PHASE3_METRICS = ["service_level", "starvations", "congestions", "functional_ratio_end"]

df3 = load_eval_results(list(PHASE3_RUN_LOGS.keys()), label_override=PHASE3_RUN_LOGS)
print(f"Loaded {len(df3)} seed rows from {df3['label'].nunique() if not df3.empty else 0} variants")
df3[["label", "seed", "service_level", "starvations", "congestions"]].head() if not df3.empty else print("No data yet.")

In [ ]:
# ── Phase 3: Summary table and bar chart ─────────────────────────────────
if not df3.empty:
    display(eval_summary_table(df3, PHASE3_METRICS, sort_by="service_level"))
    fig = plot_eval_bars(df3, PHASE3_METRICS, sort_by="service_level")
    plt.show()
else:
    print("No data loaded yet \u2014 populate PHASE3_RUN_LOGS and re-run.")

In [ ]:
# ── Phase 3: Lock reward weights ──────────────────────────────────────────
LOCKED_WS = -1.0   # ← weight_starvation
LOCKED_WC = -1.0   # ← weight_congestion
print(f"Locked:  ws = {LOCKED_WS},  wc = {LOCKED_WC}")

---
## Phase 4 — Rollout Parameters (H, S, R)

Coordinate-descent evaluation with the locked trained VFA — no retraining.
Uses `run_simulation_ingvild.py`, which has all required flags and warmup support
(7-day warmup + 21-day evaluation = `--warmup-days 7 --duration 504`).

**Step 1**: vary H ∈ {30, 60, 90} min with S=5, R=5 fixed.

```bash
# Template — run once per H value, changing --lookahead
python policies/sjovik_sund/run_simulation_ingvild.py \
  --policy hybrid \
  --vfa-model <PATH_TO_LOCKED_VFA_PKL> \
  --lookahead 60 --num-scenarios 5 --n-routing 5 \
  --warmup-days 7 --duration 504 \
  --nsims 10 --seed 42 \
  --log-files results daily decisions
```

The run log folder is named `Hybrid_<exp>_H{H}_S{S}_R{R}_...` — point `PHASE4_RUN_LOGS` to it.

**Step 2**: fix best H, vary S ∈ {3, 5, 10} (change `--num-scenarios`).  
**Step 3**: fix best H + S, vary R ∈ {3, 5, 10} (change `--n-routing`).

In [ ]:
# ── Phase 4 config ────────────────────────────────────────────────────────
# Each entry: run_log_path → display label.  Use consistent naming: H{h} S{s} R{r}
PHASE4_RUN_LOGS = {
    # Step 1 — vary H (S=5, R=5)
    "run_logs/tuning/phase4_H30_S5_R5":  "H=30, S=5, R=5",
    "run_logs/tuning/phase4_H60_S5_R5":  "H=60, S=5, R=5",
    "run_logs/tuning/phase4_H90_S5_R5":  "H=90, S=5, R=5",
    # Step 2 — vary S (uncomment after locking H)
    # "run_logs/tuning/phase4_H60_S3_R5":  "H=60, S=3, R=5",
    # "run_logs/tuning/phase4_H60_S10_R5": "H=60, S=10, R=5",
    # Step 3 — vary R (uncomment after locking H + S)
    # "run_logs/tuning/phase4_H60_S5_R3":  "H=60, S=5, R=3",
    # "run_logs/tuning/phase4_H60_S5_R10": "H=60, S=5, R=10",
}

# Metrics — add/remove columns here as needed
PHASE4_METRICS = [
    "service_level",
    "starvations",
    "congestions",
    "functional_ratio_end",   # FR: fleet functional ratio at end of simulation
    "maintenance_pct",        # (onsite_repairs + depot_pickups) / all maint+reb actions
]

df4 = load_eval_results(list(PHASE4_RUN_LOGS.keys()), label_override=PHASE4_RUN_LOGS)
print(f"Loaded {len(df4)} seed rows from {df4['label'].nunique() if not df4.empty else 0} configs")
df4[["label", "seed", "service_level", "starvations", "congestions"]].head(10) if not df4.empty else print("No data yet.")

In [ ]:
# ── Phase 4: Summary table ────────────────────────────────────────────────
if not df4.empty:
    display(eval_summary_table(df4, PHASE4_METRICS, sort_by="service_level"))
else:
    print("No data yet \u2014 populate PHASE4_RUN_LOGS and re-run.")

In [ ]:
# ── Phase 4: Bar chart ────────────────────────────────────────────────────
if not df4.empty:
    fig = plot_eval_bars(df4, PHASE4_METRICS, sort_by="service_level")
    plt.show()

In [ ]:
# ── Phase 4: Lock rollout parameters ─────────────────────────────────────
LOCKED_LOOKAHEAD  = 60   # minutes
LOCKED_SCENARIOS  = 5
LOCKED_ROUTING    = 5
print(f"Locked:  H = {LOCKED_LOOKAHEAD} min,  S = {LOCKED_SCENARIOS},  R = {LOCKED_ROUTING}")

---
## Final Configuration Summary

In [ ]:
summary_config = pd.DataFrame([
    {"Parameter": r"Feature set",          "Value": "EX15 FleetBrokenGlobal",   "Phase": "\u2014"},
    {"Parameter": r"Learning rate \u03b1", "Value": str(LOCKED_ALPHA),           "Phase": "1"},
    {"Parameter": r"Discount factor \u03b3","Value": str(LOCKED_GAMMA),          "Phase": "2"},
    {"Parameter": r"Starvation weight ws", "Value": str(LOCKED_WS),              "Phase": "3"},
    {"Parameter": r"Congestion weight wc", "Value": str(LOCKED_WC),              "Phase": "3"},
    {"Parameter": r"Lookahead H (min)",    "Value": str(LOCKED_LOOKAHEAD),       "Phase": "4"},
    {"Parameter": r"Scenarios S",          "Value": str(LOCKED_SCENARIOS),       "Phase": "4"},
    {"Parameter": r"Routing candidates R", "Value": str(LOCKED_ROUTING),         "Phase": "4"},
]).set_index("Parameter")

summary_config.style.set_caption("Final tuned configuration \u2014 EX15 FleetBrokenGlobal")